# Cybershuttle SDK -  Molecular Dynamics
> Plan, distribute, monitor, and analyze NAMD experiments across HPC runtimes.

This notebook demonstrates how to plan, launch, monitor, and analyze **NAMD** experiments with replicas through the Cybershuttle SDK.

## 1. Preliminaries

### 1.1 Install the SDK

Install the Airavata Python SDK to orchestrate NAMD experiments directly from JupyterLab. Requires Python 3.10+.

In [1]:
%pip install -qU "airavata-python-sdk[notebook]==2.2.3"

Note: you may need to restart the kernel to use updated packages.


### 1.2 Point the SDK to Cybershuttle

Configure which Cybershuttle deployment you’re targeting. Defaults to Production.

In [2]:
import os

# Production
os.environ['AUTH_SERVER_URL'] = "https://auth.cybershuttle.org"
os.environ['API_SERVER_HOSTNAME'] = "api.gateway.cybershuttle.org"
os.environ['GATEWAY_URL'] = "https://gateway.cybershuttle.org"
os.environ['STORAGE_RESOURCE_HOST'] = "gateway.cybershuttle.org"

# # Development
# os.environ['AUTH_SERVER_URL'] = "https://auth.dev.cybershuttle.org"
# os.environ['API_SERVER_HOSTNAME'] = "api.dev.cybershuttle.org"
# os.environ['GATEWAY_URL'] = "https://gateway.dev.cybershuttle.org"
# os.environ['STORAGE_RESOURCE_HOST'] = "gateway.dev.cybershuttle.org"

### 1.3 Import Packages

Load the core API and MD app bindings (NAMD, AMBER, GROMACS, etc.).

In [3]:
import airavata_experiments as ae

### 1.4 Authenticate

Call `ae.login()` to generate a one-time login link to sign in with your institution or email.

In [5]:
ae.login()

### 1.5 List Available Runtimes
Call `ae.find_runtimes()` to list all HPC resources you have access to.

In [6]:
runtimes = ae.find_runtimes(group="MDWorkshop")
ae.display(runtimes)

,cluster,category,queue_name,node_count,cpu_count,gpu_count,walltime,group
id,,,,,,,,
remote,expanse,GPU,gpu-shared,1,39,1,30,MDWorkshop
remote,NCSADelta,GPU,gpuA100x4,100,128,1,30,MDWorkshop


## 2. Run a NAMD Experiment

We preloaded a sample pull‐simulation under `data/namd`.
You can also bring in (drag-drop) your own files to run experiments.

```bash
data/namd
├── b4pull.pdb
├── b4pull.restart.coor
├── b4pull.restart.vel
├── b4pull.restart.xsc
├── par_all36_water.prm
├── par_all36m_prot.prm
├── pull_cpu.conf
├── structure.pdb
└── structure.psf

```

### 2.1 Create Experiment

Define your NAMD experiment by calling `ae.md.NAMD.initialize()` and providing the paths to your `.conf`, `.pdb`, `.psf`, and other files.
> If your IDE supports auto-completion, it will show you the method signature.

```python
def initialize(
    name: str,
    config_file: str,
    pdb_file: str,
    psf_file: str,
    ffp_files: list[str],
    other_files: list[str] = [],
    parallelism: Literal['CPU', 'GPU'] = "CPU",
    num_replicas: int = 1
) -> Experiment[ExperimentApp]
```

To add replica runs, call `exp.add_run()` once per replica -- you can optionally specify runtime and resource constraints for each run.

To perform parameter sweeps, iterate over your parameter space and call `exp.add_run()` with each parameter set as keyword arguments.

In [7]:
exp = ae.md.NAMD.initialize(
    name="SMD",
    config_file="data/namd/pull.conf",
    pdb_file="data/namd/structure.pdb",
    psf_file="data/namd/structure.psf",
    ffp_files=[
      "data/namd/par_all36_water.prm",
      "data/namd/par_all36m_prot.prm"
    ],
    other_files=[
      "data/namd/b4pull.pdb",
      "data/namd/b4pull.restart.coor",
      "data/namd/b4pull.restart.vel",
      "data/namd/b4pull.restart.xsc",
    ],
    parallelism="GPU",
)
runtimes = ae.find_runtimes(group="MDWorkshop", cluster="NCSADelta")
for _ in range(2):
    exp.add_run(use=runtimes, cpus=8, nodes=1, walltime=60)

Task created. (1 tasks in total)
Task created. (2 tasks in total)


Call `ae.display(exp)` to print the experiment details.

In [8]:
ae.display(exp)

,application,num_tasks,config_file,pdb_file,psf_file,ffp_files,parallelism,other_files,num_replicas
name,,,,,,,,,
SMD,NAMD,2,data/namd/pull.conf,data/namd/structure.pdb,data/namd/structure.psf,"data/namd/par_all36_water.prm, data/namd/par_a...",GPU,"data/namd/b4pull.pdb, data/namd/b4pull.restart...",1


### 2.2 Build Execution Plan

Call `exp.plan()` to upload your inputs to Cybershuttle and create a reproducible plan that's runnable from anywhere.

Call `ae.display(plan)` to print the plan details.

In [9]:
plan = exp.plan()
ae.display(plan)

Plan saved: 407e488b-2b9f-421c-940e-239c5f217890


Call `plan.export()` to export the plan (as JSON) for future reference.

In [10]:
plan.export("plan_gpu_2.json")

### 2.3 Launch Experiment

Call `plan.launch()` to launch the experiment.
The SDK will record tracking metadata (job IDs, working directories, etc.) and update the plan state in Cybershuttle.
> If you exported your plan before, rerun `plan.export()` to refresh your local JSON.

In [11]:
plan.launch()
plan.export("plan_gpu_2.json")

Preparing to launch...
Launching tasks...
[Task] Executing SMD_1191 on Remote(args={'cluster': 'NCSADelta', 'category': 'GPU', 'queue_name': 'gpuA100x4', 'node_count': 1, 'cpu_count': 8, 'gpu_count': 1, 'walltime': 60, 'group': 'MDWorkshop'})
[Remote] Creating Experiment: name=SMD_1191
[AV] Preprocessing args...
[AV] Validating args...
[AV] Setting up runtime params...
[AV] Setting up application interface...
[AV] Setting up experiment...
[AV] Setting up experiment directory...
[AV] exp_dir: /Default_Project/SMD_1191_2025_08_05_19_56_03
[AV] abs_path: /var/www/portals/gateway-user-data/default/spamidig@gatech.edu/Default_Project/SMD_1191_2025_08_05_19_56_03/
[AV] Setting up computation resource scheduling...
[AV] Setting up experiment inputs...


Output()

[AV] * agent_id=0f6a606b-85e9-456b-b748-45921c05ee9b
[AV] * Constraints-PDB=None
[AV] * Continue_from_Previous_Run?=None
[AV] * Coordinates-PDB-File=airavata-dp://5130c861-6851-4dfc-ae71-d54c3d4172b8
[AV] * Execution_Type=GPU
[AV] * FF-Parameter-Files=airavata-dp://f8f367f8-4527-488b-a80e-2446c97e87a2,airavata-dp://f66b4266-328d-4a52-afba-1987a0d6c9f0
[AV] * GPU Resource Warning=Change the Queue from CPU to a GPU one (A100X4 or A100X8 or MI100X4) using 'Select a Queue' clicking on 'Settings for Queue' below
[AV] * MD-Instructions-Input=airavata-dp://fa41aa61-e4d0-420b-9a87-db86eae7a276
[AV] * Number of Replicas=1
[AV] * Optional_Inputs=airavata-dp://ce4da38e-ddbb-4522-93a3-b00d5e6a3df8,airavata-dp://2735092d-9380-43a8-9e0d-9db436642f77,airavata-dp://31b7e8cd-8f6b-434d-95fd-0a9bbc974a67,airavata-dp://c9b5b9ad-9b79-4a6d-b653-24330da5c22e
[AV] * Previous_JobID=None
[AV] * Protein-Structure-File_PSF=airavata-dp://0b50c57e-ca78-4f94-949e-f0eec3e6e55f
[AV] * Replicate?=None
[AV] * Restart_Re

[AV] Creating experiment...
[AV] Experiment SMD_1191 CREATED with id: SMD_1191_bc9d9af6-c0b9-4309-aad2-42cc8704d17f
[AV] Experiment SMD_1191 STARTED with id: SMD_1191_bc9d9af6-c0b9-4309-aad2-42cc8704d17f
[AV] Experiment SMD_1191 WAITING until experiment begins...
[AV] Experiment SMD_1191 EXECUTING with pid: PROCESS_b06a4a4c-1caf-472e-a28b-c087f37a85f5
[AV] Experiment SMD_1191 WAITING until task begins...
[AV] Experiment SMD_1191 - Task QUEUED with id: 11337811
[Remote] Experiment Launched: id=SMD_1191_bc9d9af6-c0b9-4309-aad2-42cc8704d17f
[Task] Executing SMD_D169 on Remote(args={'cluster': 'NCSADelta', 'category': 'GPU', 'queue_name': 'gpuA100x4', 'node_count': 1, 'cpu_count': 8, 'gpu_count': 1, 'walltime': 60, 'group': 'MDWorkshop'})
[Remote] Creating Experiment: name=SMD_D169
[AV] Preprocessing args...
[AV] Validating args...
[AV] Setting up runtime params...
[AV] Setting up application interface...
[AV] Setting up experiment...
[AV] Setting up experiment directory...
[AV] exp_dir: /

Output()

[AV] * agent_id=2fe8b445-fa1b-4634-8267-233a736a6505
[AV] * Constraints-PDB=None
[AV] * Continue_from_Previous_Run?=None
[AV] * Coordinates-PDB-File=airavata-dp://aed93f08-fcaf-465a-9fe1-fd7d569c6980
[AV] * Execution_Type=GPU
[AV] * FF-Parameter-Files=airavata-dp://96415fce-668c-49b9-b802-e9fa56735eda,airavata-dp://1609db2c-91b6-4bbf-8534-55ca1bcd4bf6
[AV] * GPU Resource Warning=Change the Queue from CPU to a GPU one (A100X4 or A100X8 or MI100X4) using 'Select a Queue' clicking on 'Settings for Queue' below
[AV] * MD-Instructions-Input=airavata-dp://e952359d-7414-4953-a40f-19f6ccb5a821
[AV] * Number of Replicas=1
[AV] * Optional_Inputs=airavata-dp://3ad796f7-c29f-49bd-aebc-9fef65d72be7,airavata-dp://db336cf1-4da5-42b8-8791-beea2fa388a0,airavata-dp://c1467fcc-84ae-4820-b20c-3ad8ce112b52,airavata-dp://dbe67cce-ab15-4b6f-9ff1-c70312750dc6
[AV] * Previous_JobID=None
[AV] * Protein-Structure-File_PSF=airavata-dp://ce15bb5c-8a83-45c2-bb7a-c21d7cd81c98
[AV] * Replicate?=None
[AV] * Restart_Re

[AV] Creating experiment...
[AV] Experiment SMD_D169 CREATED with id: SMD_D169_961becfc-e0bd-44b0-af83-0908c7554bc6
[AV] Experiment SMD_D169 STARTED with id: SMD_D169_961becfc-e0bd-44b0-af83-0908c7554bc6
[AV] Experiment SMD_D169 WAITING until experiment begins...
[AV] Experiment SMD_D169 EXECUTING with pid: PROCESS_da536293-6b63-4f57-8890-5995d036c757
[AV] Experiment SMD_D169 WAITING until task begins...
[AV] Experiment SMD_D169 - Task UNKNOWN with id: 11337813
[Remote] Experiment Launched: id=SMD_D169_961becfc-e0bd-44b0-af83-0908c7554bc6
Plan updated: 407e488b-2b9f-421c-940e-239c5f217890


### 2.4 Monitor Experiment

Call `plan.status()` to check the current state of the experiment and its runs. You can poll for experiment completion by calling this periodically.

In [12]:
plan.export("plan_gpu_2.json")

In [13]:
plan.status()

Plan 407e488b-2b9f-421c-940e-239c5f217890 (2 tasks):
* SMD_1191: 11337811: ACTIVE
* SMD_D169: 11337813: ACTIVE


You can loop over `plan.tasks` to interact with each run in real time.
The SDK provides helper functions to print metadata, list files, transfer files, preview file content, and run shell commands on the fly.

Each `task` object has several helper functions to perform file operations within its context.

* `task.ls()` - list all remote files (inputs, outputs, logs, etc.)
* `task.upload(<local_path>, <remote_path>)` - upload a local file to remote
* `task.cat(<remote_path>)` - displays contents of a remote file
* `task.download(<remote_path>, <local_path>)` - fetch a remote file to local

In [ ]:
for task in plan.tasks:
    print(task.name, task.pid, task.workdir)
    display(task.ls())                                      # list files
    task.upload("data/sample.txt")                          # upload sample.txt
    display(task.cat("sample.txt"))                         # preview sample.txt
    task.exec("cat sample.txt | xargs wc -l > count.log")   # generate count.log
    display(task.cat("count.log"))                          # preview count.log
    task.download("count.log", f"./results_{task.name}")    # download count.log

Call `plan.wait_for_completion()` to block further execution until the experiment completes.

In [ ]:
plan.wait_for_completion()

Alternatively, you can call `plan.stop()` to stop the entire experiment (all runs). To stop specific runs, pass their indices as a `runs` argument.

e.g., `plan.stop(runs=[a,b])` stops only the $a^{th}$ and $b^{th}$ runs; others, if any, will continue to run.

## 3. Analyze Experiment Runs


### 3.1 List All Experiments

Call `ae.plan.query()` to retrieve all plans you created through the SDK.

In [14]:
plans = ae.plan.query()
ae.display(plans)

You can find a plan, record its id, and call `ae.plan.load(plan_id)` with that id to load it into memory.
Alternatively, if you exported a plan before, call `ae.plan.load_json(path_to_plan)` to load that instead.

In [15]:
plan = ae.plan.load_json("plan_gpu_2.json")
plan = ae.plan.load(plan.id)
ae.display(plan)

### 3.2 Process Experiment Results Interactively

You can launch an interactive job where the plan was executed, without requiring extra configuration.
The `--state=<path/to/plan/file>` argument will bring those results into the interactive job.
All results will be stored in a `<project_name>_results`

In [16]:
import airavata_jupyter_magic

%request_runtime hpc --file=cybershuttle.yml --walltime=60 --plan=plan_gpu_2.json
%wait_for_runtime hpc --live
%switch_runtime hpc


Loaded airavata_jupyter_magic (2.2.3) 
(current runtime = local)

  %authenticate                              -- Authenticate to access high-performance runtimes.
  %request_runtime <rt> [args]               -- Request a runtime named <rt> with configuration <args>.
                                                Call multiple times to request multiple runtimes.
  %restart_runtime <rt>                      -- Restart runtime <rt> if it hangs. This will clear all variables.
  %stop_runtime <rt>                         -- Stop runtime <rt> when no longer needed.
  %wait_for_runtime <rt>                     -- Wait for runtime <rt> to be ready.
  %switch_runtime <rt>                       -- Switch the active runtime to <rt>. All subsequent cells will run here.
  %%run_on <rt>                              -- Force a cell to always execute on <rt>, regardless of the active runtime.
  %stat_runtime <rt>                         -- Show the status of runtime <rt>.
  %copy_data source=<r1:f1

Output()

local:/tmp/connection_rch7kbjr.json --> hpc:connection_rch7kbjr.json... [200]
started proc_name=hpc_kernel on rt=hpc. pid=1358038
forwarding ports=[24765, 24766, 24767, 24768, 24769]
hpc:24765 -> access via 18.118.140.230:10005
hpc:24766 -> access via 18.118.140.230:10006
hpc:24767 -> access via 18.118.140.230:10007
hpc:24768 -> access via 18.118.140.230:10008
hpc:24769 -> access via 18.118.140.230:10009
started ipykernel tunnels for hpc at 18.118.140.230
started ipykernel client for hpc
Remote Jupyter kernel launched and connected for runtime=hpc.
[av] linked ../PROCESS_b06a4a4c-1caf-472e-a28b-c087f37a85f5 -> SMD_1191
[av] linked ../PROCESS_da536293-6b63-4f57-8890-5995d036c757 -> SMD_D169
Switched to runtime=hpc.


In [17]:
%switch_runtime local
plan = ae.plan.load_json("plan_gpu_2.json")
plan = ae.plan.load(plan.id)
ae.display(plan)

Switched to runtime=local.


# Upload analysis script using %copy_data 

In [18]:
%copy_data source=local:plan_gpu_2.json target=hpc:plan_gpu_2.json

copying local:plan_gpu_2.json to hpc:plan_gpu_2.json
local:plan_gpu_2.json --> hpc:plan_gpu_2.json... [200]


In [19]:
#!copy_data source=test.tcl target=hpc:SMD_40AB/test.tcl
%switch_runtime hpc
plan = ae.plan.load_json("plan_gpu_2.json")
plan = ae.plan.load(plan.id)
for task in plan.tasks:
    print(task.name, task.pid, task.workdir)
    display(task.ls())                                      # list files
    task.upload("test.tcl")                          # upload sample.txt
    display(task.cat("test.tcl"))                         # preview sample.txt
    task.exec("cat test.tcl")   # generate count.log
    #display(task.cat("count.log"))                          # preview count.log
    #task.download("count.log", f"./results_{task.name}")    # download count.log

Switched to runtime=hpc.
SMD_1191 PROCESS_b06a4a4c-1caf-472e-a28b-c087f37a85f5 /Default_Project/SMD_1191_2025_08_05_19_56_03


['1/A636768135',
 '1/FFTW_NAMD_3.0b3_Linux-x86_64-multicore-CUDA.txt',
 '1/b4pull.pdb',
 '1/b4pull.restart.coor',
 '1/b4pull.restart.vel',
 '1/b4pull.restart.xsc',
 '1/par_all36_water.prm',
 '1/par_all36m_prot.prm',
 '1/pull.conf',
 '1/pull.conf.err',
 '1/pull.conf.out',
 '1/structure.pdb',
 '1/structure.psf',
 '1/system.1.4.dcd',
 '1/system.1.4.xst',
 '1/system.1.4r.coor',
 '1/system.1.4r.coor.old',
 '1/system.1.4r.vel',
 '1/system.1.4r.vel.old',
 '1/system.1.4r.xsc',
 '1/system.1.4r.xsc.old',
 'A636768135',
 'NAMD.stderr',
 'NAMD.stdout',
 'airavata-agent',
 'b4pull.pdb',
 'b4pull.restart.coor',
 'b4pull.restart.vel',
 'b4pull.restart.xsc',
 'job_464076274.slurm',
 'kernel.py',
 'micromamba',
 'par_all36_water.prm',
 'par_all36m_prot.prm',
 'pull.conf',
 'structure.pdb',
 'structure.psf',
 '']

b'mol new structure.psf\nmol addfile 1_system.1.4r.coor\nset sel [ atomselect top protein ]\n$sel num\nquit'

SMD_D169 PROCESS_da536293-6b63-4f57-8890-5995d036c757 /Default_Project/SMD_D169_2025_08_05_19_56_29


['1/A583535747',
 '1/FFTW_NAMD_3.0b3_Linux-x86_64-multicore-CUDA.txt',
 '1/b4pull.pdb',
 '1/b4pull.restart.coor',
 '1/b4pull.restart.vel',
 '1/b4pull.restart.xsc',
 '1/par_all36_water.prm',
 '1/par_all36m_prot.prm',
 '1/pull.conf',
 '1/pull.conf.err',
 '1/pull.conf.out',
 '1/structure.pdb',
 '1/structure.psf',
 '1/system.1.4.dcd',
 '1/system.1.4.xst',
 '1/system.1.4r.coor',
 '1/system.1.4r.coor.old',
 '1/system.1.4r.vel',
 '1/system.1.4r.vel.old',
 '1/system.1.4r.xsc',
 '1/system.1.4r.xsc.old',
 'A583535747',
 'NAMD.stderr',
 'NAMD.stdout',
 'airavata-agent',
 'b4pull.pdb',
 'b4pull.restart.coor',
 'b4pull.restart.vel',
 'b4pull.restart.xsc',
 'job_2104783577.slurm',
 'kernel.py',
 'micromamba',
 'par_all36_water.prm',
 'par_all36m_prot.prm',
 'pull.conf',
 'structure.pdb',
 'structure.psf',
 '']

b'mol new structure.psf\nmol addfile 1_system.1.4r.coor\nset sel [ atomselect top protein ]\n$sel num\nquit'

In [ ]:
#for task in plan.tasks:
%copy_data source=local:test.tcl target=hpc:SMD_1191/test.tcl
%copy_data source=local:test.tcl target=hpc:SMD_D169/test.tcl


In [25]:
%switch_runtime local
plan = ae.plan.load_json("plan_gpu_2.json")
plan = ae.plan.load(plan.id)
ae.display(plan)
%copy_data source=local:test.tcl target=hpc:SMD_1191/test.tcl
%copy_data source=local:test.tcl target=hpc:SMD_D169/test.tcl
%switch_runtime hpc

Switched to runtime=local.


copying local:test.tcl to hpc:SMD_1191/test.tcl
local:test.tcl --> hpc:SMD_1191/test.tcl... [200]
copying local:test.tcl to hpc:SMD_D169/test.tcl
local:test.tcl --> hpc:SMD_D169/test.tcl... [200]
Switched to runtime=hpc.


In [27]:
#!copy_data source=test.tcl target=hpc:SMD_40AB/test.tcl
%switch_runtime hpc
plan = ae.plan.load_json("plan_gpu_2.json")
plan = ae.plan.load(plan.id)
for task in plan.tasks:
    print(task.name, task.pid, task.workdir)
    display(task.ls())                                      # list files
    task.upload("test.tcl")                          # upload sample.txt
    display(task.cat("test.tcl"))                         # preview sample.txt
    task.exec("cat test.tcl")   # generate count.log
    #display(task.cat("count.log"))                          # preview count.log
    #task.download("count.log", f"./results_{task.name}")    # download count.log

Switched to runtime=hpc.
SMD_1191 PROCESS_b06a4a4c-1caf-472e-a28b-c087f37a85f5 /Default_Project/SMD_1191_2025_08_05_19_56_03


['1_system.1.4.coor',
 '1_pull.conf.out',
 '1_b4pull.restart.coor',
 '1_b4pull.restart.xsc',
 '1_b4pull.restart.vel',
 'par_all36_water.prm',
 'b4pull.restart.vel',
 'NAMD.stdout',
 '1_system.1.4.dcd',
 'par_all36m_prot.prm',
 '1_b4pull.pdb',
 'b4pull.pdb',
 'b4pull.restart.xsc',
 'pull.conf',
 '1_structure.pdb',
 '1_structure.psf',
 'b4pull.restart.coor',
 'structure.pdb',
 'structure.psf',
 '1_system.1.4r.coor',
 'NAMD.stderr']

Output()

b'mol new structure.psf\nmol addfile 1_system.1.4r.coor\nset sel [ atomselect top protein ]\n$sel num\nquit'

[Remote] Failed to execute command: Exception('Agent not found')
SMD_D169 PROCESS_da536293-6b63-4f57-8890-5995d036c757 /Default_Project/SMD_D169_2025_08_05_19_56_29


['1_system.1.4.coor',
 '1_pull.conf.out',
 '1_b4pull.restart.coor',
 '1_b4pull.restart.xsc',
 '1_b4pull.restart.vel',
 'par_all36_water.prm',
 'b4pull.restart.vel',
 'NAMD.stdout',
 '1_system.1.4.dcd',
 'par_all36m_prot.prm',
 '1_b4pull.pdb',
 'b4pull.pdb',
 'b4pull.restart.xsc',
 'pull.conf',
 '1_structure.pdb',
 '1_structure.psf',
 'b4pull.restart.coor',
 'structure.pdb',
 'structure.psf',
 '1_system.1.4r.coor',
 'NAMD.stderr']

Output()

b'mol new structure.psf\nmol addfile 1_system.1.4r.coor\nset sel [ atomselect top protein ]\n$sel num\nquit'

[Remote] Failed to execute command: Exception('Agent not found')


In [35]:
%%bash
ls -lt
module load vmd
#for task in plan.tasks:
cd SMD_1191 
ls -lt test.tcl
cat test.tcl    
vmd -dispdev text -e test.tcl
pwd
 

cd SMD_D169
ls -lt test.tcl
cat test.tcl    
vmd -dispdev text -e test.tcl
cd ..

    
#echo $PWD
#ls -lrt .
#tail -n 100 1_pull.conf.out


executing cell on hpc...
waiting for cell to finish on hpc...
total 43972
-rw-r-----+ 1 svcscigapgwuser delta_bbol     3507 Aug  5 15:01 plan_gpu_2.json
lrwxrwxrwx  1 svcscigapgwuser delta_bbol       47 Aug  5 15:01 SMD_1191 -> ../PROCESS_b06a4a4c-1caf-472e-a28b-c087f37a85f5
lrwxrwxrwx  1 svcscigapgwuser delta_bbol       47 Aug  5 15:01 SMD_D169 -> ../PROCESS_da536293-6b63-4f57-8890-5995d036c757
-rw-rw----+ 1 svcscigapgwuser delta_bbol     5991 Aug  5 15:01 AiravataAgent.stderr
-rw-rw----+ 1 svcscigapgwuser delta_bbol    16490 Aug  5 15:01 AiravataAgent.stdout
-rw-r-----+ 1 svcscigapgwuser delta_bbol      232 Aug  5 15:00 connection_rch7kbjr.json
lrwxrwxrwx  1 svcscigapgwuser delta_bbol       43 Aug  5 14:59 application -> /u/svcscigapgwuser/cybershuttle/application
-rw-rw----+ 1 svcscigapgwuser delta_bbol       29 Aug  5 14:59 A2039372282
-rw-r-----+ 1 svcscigapgwuser delta_bbol     1468 Aug  5 14:59 job_1502157603.slurm
-rwxrwx--x+ 1 svcscigapgwuser delta_bbol 28466332 May  6 19:20 a

/sw/external/vmd/lib/vmd_LINUXAMD64: /lib64/libGL.so.1: no version information available (required by /sw/external/vmd/lib/vmd_LINUXAMD64)


quitInfo) VMD for LINUXAMD64, version 1.9.3 (November 30, 2016)
Info) http://www.ks.uiuc.edu/Research/vmd/                         
Info) Email questions and bug reports to vmd@ks.uiuc.edu           
Info) Please include this reference in published work using VMD:   
Info)    Humphrey, W., Dalke, A. and Schulten, K., `VMD - Visual   
Info)    Molecular Dynamics', J. Molec. Graphics 1996, 14.1, 33-38.
Info) -------------------------------------------------------------
Info) Multithreading available, 64 CPUs detected.
Info)   CPU features: SSE2 AVX AVX2 FMA 
Info) Free system memory: 243GB (96%)
Info) Creating CUDA device pool and initializing hardware...
Info) Detected 1 available CUDA accelerator:
Info) [0] NVIDIA A100-SXM4-40GB 108 SM_8.0 @ 1.41 GHz, 39GB RAM, AE5, ZCP
Info) Detected 1 available TachyonL/OptiX ray tracing accelerator
Info)   Compiling 1 OptiX shaders on 1 target GPU...
Info) Dynamically loaded 2 plugins in directory:
Info) /sw/external/vmd/lib/plugins/LINUXAMD64/molfi

In [31]:
%%bash
cd SMD_D169
ls -lt

executing cell on hpc...
waiting for cell to finish on hpc...
total 1257108
-rw-rw----+ 1 svcscigapgwuser delta_bbol        101 Aug  5 15:13 test.tcl
-rw-rw----+ 1 svcscigapgwuser delta_bbol     558996 Aug  5 15:06 1_pull.conf.out
-rw-r-----+ 1 svcscigapgwuser delta_bbol    2332708 Aug  5 15:06 1_system.1.4.vel
-rw-rw----+ 1 svcscigapgwuser delta_bbol      25400 Aug  5 15:06 NAMD.stderr
-rw-r-----+ 1 svcscigapgwuser delta_bbol    2332708 Aug  5 15:06 1_system.1.4.coor
-rw-r-----+ 1 svcscigapgwuser delta_bbol 1166432276 Aug  5 15:06 1_system.1.4.dcd
-rw-r-----+ 1 svcscigapgwuser delta_bbol        217 Aug  5 15:06 1_system.1.4.xsc
-rw-r-----+ 1 svcscigapgwuser delta_bbol     105983 Aug  5 15:06 1_system.1.4.xst
-rw-r-----+ 1 svcscigapgwuser delta_bbol    2332708 Aug  5 15:06 1_system.1.4r.coor
-rw-r-----+ 1 svcscigapgwuser delta_bbol    2332708 Aug  5 15:06 1_system.1.4r.vel
-rw-r-----+ 1 svcscigapgwuser delta_bbol        218 Aug  5 15:06 1_system.1.4r.xsc
-rw-r-----+ 1 svcscigapgwuser d

In [32]:
%%bash 
cd SMD_D169
tail -n 100 1_pull.conf.out

executing cell on hpc...
waiting for cell to finish on hpc...
SMD  492500 -2.05008 9.96229 62.4281 0 0 176.457
ENERGY:  492500      1064.7697      2895.6967      3491.5594       184.8720        -365266.8956     31421.7813         0.0000        22.3963     59223.8514        -266961.9690       302.0712   -326185.8203   -230438.4267         0.6041           -102.8704       -77.3136    954432.0011      1432.0498       727.9925

WRITING COORDINATES TO DCD FILE system.1.4.dcd AT STEP 492500
SMD  493000 -1.47226 10.3331 62.7181 0 0 174.056
ENERGY:  493000      1050.6478      2906.7401      3485.2732       195.0373        -364805.0778     31526.0395         0.0000        21.7909     58727.1373        -266892.4117       299.5377   -325619.5490   -230022.5763         0.5991             87.4185       117.3127    954432.0011      1446.1078       741.5563

WRITING COORDINATES TO DCD FILE system.1.4.dcd AT STEP 493000
SMD  493500 -2.06336 9.94734 62.6178 0 0 175.559
ENERGY:  493500      1062.6093   

In [ ]:
plan.stop()
plan.stop(runs=[0])

In [ ]:
%stop_runtime hpc